In [2]:
!pip install PyPDF2

  Using cached pypdf2-3.0.1-py3-none-any.whl.metadata (6.8 kB)
Using cached pypdf2-3.0.1-py3-none-any.whl (232 kB)


In [1]:
import os
import re
import torch
import numpy as np
import pandas as pd
from PyPDF2 import PdfReader
from pathlib import Path
import spacy
from pytorch_pretrained_bert import BertTokenizer, BertForTokenClassification
from keras.preprocessing.sequence import pad_sequences

In [2]:
# Load spaCy model
nlp = spacy.load("en_core_web_sm")

In [3]:
# Adding '\n' to the default spacy tokenizer as in your training code
prefixes = ('\n',) + tuple(nlp.Defaults.prefixes)
prefix_regex = spacy.util.compile_prefix_regex(prefixes)
nlp.tokenizer.prefix_search = prefix_regex.search

In [4]:
# Define entity types as in your training code
entity_dict = {
    'NAME': 'Name',
    'CLG': 'College Name',
    'DEG': 'Degree',
    'GRADYEAR': 'Graduation Year',
    'YOE': 'Years of Experience',
    'COMPANY': 'Companies worked at',
    'DESIG': 'Designation',
    'SKILLS': 'Skills',
    'LOC': 'Location',
    'EMAIL': 'Email Address'
}
# Reverse mapping for display
tag_to_display = {v: k for k, v in entity_dict.items()}

In [5]:
def pdf_to_text(pdf_path):
    """
    Convert PDF to text using PyPDF2
    Args:
        pdf_path (str): Path to the PDF file
    Returns:
        str: Extracted text from the PDF
    """
    print(f"Converting PDF to text: {pdf_path}")
    try:
        # Check if file exists
        if not os.path.exists(pdf_path):
            raise FileNotFoundError(f"PDF file not found: {pdf_path}")
        # Open the PDF file
        reader = PdfReader(pdf_path)
        # Get the number of pages
        num_pages = len(reader.pages)
        print(f"PDF loaded successfully with {num_pages} pages")
        # Extract text from each page
        full_text = ""
        for page_num in range(num_pages):
            page = reader.pages[page_num]
            full_text += page.extract_text() + "\n"
        print("Text extraction complete")
        return full_text
    except Exception as e:
        print(f"Error extracting text from PDF: {str(e)}")
        return ""

In [6]:
def clean_resume_text(text):
    """
    Clean the resume text by removing extra whitespace, special characters, etc.
    Args:
        text (str): Raw text extracted from the resume
    Returns:
        str: Cleaned text
    """
    print("Cleaning resume text...")
    # Remove extra whitespace
    cleaned = re.sub(r'\s+', ' ', text)
    # Remove special characters but keep important ones
    cleaned = re.sub(r'[^\w\s.@:,;()-]', '', cleaned)
    # Normalize line breaks
    cleaned = re.sub(r'\n+', '\n', cleaned)
    print("Text cleaning complete")
    return cleaned

In [7]:
def load_model(model_path):
    """
    Load the PyTorch BERT model for token classification
    Args:
        model_path (str): Path to the model file
    Returns:
        model: Loaded PyTorch model
        tokenizer: BERT tokenizer
        idx2tag: Index to tag mapping
    """
    print(f"Loading AI model: {model_path}")
    try:
        # Check if model file exists
        if not os.path.exists(model_path):
            raise FileNotFoundError(f"Model file not found: {model_path}")
        # Load BERT tokenizer
        tokenizer = BertTokenizer.from_pretrained('bert-base-cased', do_lower_case=False)
        # First, load the state dict to inspect it
        state_dict = torch.load(model_path, map_location=torch.device('cpu'))
        # Get the number of labels from the classifier weight shape
        num_labels = state_dict['classifier.weight'].size(0)
        print(f"Model has {num_labels} output classes")
        # Create a BERT config with the correct number of labels
        config = BertConfig.from_pretrained('bert-base-cased', num_labels=num_labels)
        # Create model with the correct number of labels
        model = BertForTokenClassification(config)
        # Load the state dict
        model.load_state_dict(state_dict)
        model.eval()
        # Create a mapping from index to tag
        # Since we don't know the exact tags, we'll create a generic mapping
        # The important part is to have the correct number of tags
        idx2tag = {}
        # Basic tags that are likely to be in the model
        basic_tags = ['O', '[CLS]', '[SEP]', 'X']
        # Entity types from your training code
        entity_types = ['NAME', 'CLG', 'DEG', 'GRADYEAR', 'YOE', 'COMPANY', 'DESIG', 'SKILLS', 'LOC', 'EMAIL']
        # BILOU prefixes
        prefixes = ['B-', 'I-', 'L-', 'U-']
        # Create all possible tags
        all_tags = basic_tags.copy()
        for prefix in prefixes:
            for entity in entity_types:
                all_tags.append(f"{prefix}{entity}")
        # Ensure we have at least as many tags as the model expects
        if len(all_tags) >= num_labels:
            # Use only as many tags as the model has outputs
            for i in range(num_labels):
                if i < len(all_tags):
                    idx2tag[i] = all_tags[i]
                else:
                    idx2tag[i] = f"TAG_{i}"  # Fallback for any extra tags
        else:
            # If we don't have enough tags, create generic ones
            for i in range(num_labels):
                if i < len(all_tags):
                    idx2tag[i] = all_tags[i]
                else:
                    idx2tag[i] = f"TAG_{i}"
        print("Model loaded successfully")
        return model, tokenizer, idx2tag
    except Exception as e:
        print(f"Error loading model: {str(e)}")
        return None, None, None

In [8]:
def prepare_resume_for_bert(text, tokenizer, max_len=512):
    """
    Prepare the resume text for BERT model inference
    Args:
        text (str): Cleaned resume text
        tokenizer: BERT tokenizer
        max_len (int): Maximum sequence length
    Returns:
        input_ids: Token IDs
        attention_masks: Attention masks
        token_type_ids: Token type IDs
        tokens: Original tokens
    """
    # Process text with spaCy
    doc = nlp(text)
    # Tokenize text
    tokens = []
    for token in doc:
        tokens.append(token.text)
    # Convert tokens to BERT tokens
    bert_tokens = []
    orig_to_tok_map = []
    bert_tokens.append('[CLS]')
    for token in tokens:
        orig_to_tok_map.append(len(bert_tokens))
        word_pieces = tokenizer.tokenize(token)
        bert_tokens.extend(word_pieces)
    bert_tokens.append('[SEP]')
    # Convert tokens to IDs
    input_ids = tokenizer.convert_tokens_to_ids(bert_tokens)
    # Pad sequences
    input_ids = pad_sequences([input_ids], maxlen=max_len, dtype="long", 
                             truncating="post", padding="post")[0]
    # Create attention masks
    attention_masks = [float(i > 0) for i in input_ids]
    # Create token type IDs (all 0 for single sequence)
    token_type_ids = [0] * len(input_ids)
    return input_ids, attention_masks, token_type_ids, bert_tokens

In [9]:
def extract_entities(model, input_ids, attention_masks, token_type_ids, tokens, idx2tag):
    """
    Extract entities from the resume using the model
    
    Args:
        model: BERT model
        input_ids: Token IDs
        attention_masks: Attention masks
        token_type_ids: Token type IDs
        tokens: Original tokens
        idx2tag: Index to tag mapping
        
    Returns:
        entities: List of extracted entities with confidence scores
    """
    print("Extracting entities from resume...")
    
    try:
        # Convert inputs to tensors
        input_ids_tensor = torch.tensor([input_ids]).long()
        attention_masks_tensor = torch.tensor([attention_masks]).long()
        token_type_ids_tensor = torch.tensor([token_type_ids]).long()
        
        # Get model predictions
        with torch.no_grad():
            outputs = model(input_ids_tensor, 
                           token_type_ids=token_type_ids_tensor, 
                           attention_mask=attention_masks_tensor)
            
        # Get logits and apply softmax to get probabilities
        logits = outputs[0] if isinstance(outputs, tuple) else outputs
        probs = torch.nn.functional.softmax(logits, dim=2)
        
        # Get predicted tags and confidence scores
        pred_tags = torch.argmax(probs, dim=2)[0].tolist()
        confidence_scores = torch.max(probs, dim=2)[0][0].tolist()
        
        # Convert predicted tags to tag names
        pred_tag_names = [idx2tag.get(tag_idx, 'O') for tag_idx in pred_tags]
        
        # Extract entities
        entities = []
        current_entity = None
        
        for i in range(1, min(len(tokens), len(pred_tag_names))):  # Skip [CLS]
            if tokens[i] == '[SEP]':
                break
                
            tag = pred_tag_names[i]
            confidence = confidence_scores[i]
            
            # Skip 'X' (subword) and 'O' (outside) tags
            if tag == 'X' or tag == 'O' or tag == '[CLS]' or tag == '[SEP]':
                continue
                
            # Extract entity type and position (B, I, L, U)
            if '-' in tag:
                pos, entity_type = tag.split('-', 1)
            else:
                continue
                
            # Handle different positions
            if pos == 'B' or pos == 'U':  # Beginning or Unit
                if current_entity:
                    entities.append(current_entity)
                current_entity = {
                    'type': entity_type,
                    'value': tokens[i],
                    'confidence': confidence
                }
            elif pos == 'I' or pos == 'L':  # Inside or Last
                if current_entity and current_entity['type'] == entity_type:
                    current_entity['value'] += ' ' + tokens[i]
                    # Update confidence as average
                    current_entity['confidence'] = (current_entity['confidence'] + confidence) / 2
                    
                    if pos == 'L':  # Last token of entity
                        entities.append(current_entity)
                        current_entity = None
        
        # Add the last entity if it exists
        if current_entity:
            entities.append(current_entity)
        
        # Group similar entities
        grouped_entities = {}
        for entity in entities:
            entity_type = entity['type']
            if entity_type not in grouped_entities:
                grouped_entities[entity_type] = []
            grouped_entities[entity_type].append(entity)
        
        # Clean entity values
        for entity_type, entity_list in grouped_entities.items():
            for entity in entity_list:
                # Remove special tokens and clean up
                entity['value'] = entity['value'].replace('[CLS]', '').replace('[SEP]', '').strip()
        
        print("Entity extraction complete")
        return grouped_entities
    
    except Exception as e:
        print(f"Error extracting entities: {str(e)}")
        return {}

In [10]:
def display_results(grouped_entities):
    """
    Display the extracted entities with confidence scores
    Args:
        grouped_entities: Dictionary of grouped entities
    """
    print("\n=== EXTRACTED ENTITIES ===")
    if not grouped_entities:
        print("No entities were extracted from the resume.")
        return
    total_confidence = 0
    total_entities = 0
    # Display grouped entities with confidence scores
    for entity_type, entity_list in grouped_entities.items():
        # Get the display name for the entity type
        display_name = entity_dict.get(entity_type, entity_type)
        print(f"\n{display_name}:")
        for entity in entity_list:
            confidence_percentage = f"{entity['confidence'] * 100:.2f}"
            print(f"  - {entity['value']} (Confidence: {confidence_percentage}%)")
            total_confidence += entity['confidence']
            total_entities += 1
    # Calculate overall model accuracy
    if total_entities > 0:
        average_confidence = total_confidence / total_entities
        print(f"\nOverall Model Accuracy: {average_confidence * 100:.2f}%")
    else:
        print("\nNo entities extracted")

In [11]:
def parse_resume(resume_path, model_path):
    """
    Main function to parse the resume
    
    Args:
        resume_path (str): Path to the resume PDF file
        model_path (str): Path to the model file
    """
    print("Starting resume parsing process...")
    
    try:
        # Step 1: Convert PDF to text
        resume_text = pdf_to_text(resume_path)
        if not resume_text:
            print("Failed to extract text from PDF. Please check the file.")
            return
            
        print("\nExtracted Text Sample:")
        print(resume_text[:200] + "..." if len(resume_text) > 200 else resume_text)
        
        # Step 2: Clean the text
        cleaned_text = clean_resume_text(resume_text)
        print("\nCleaned Text Sample:")
        print(cleaned_text[:200] + "..." if len(cleaned_text) > 200 else cleaned_text)
        
        # Step 3: Load the model
        model, tokenizer, idx2tag = load_model(model_path)
        if not model:
            print("Failed to load model. Please check the model file.")
            return
        
        # Step 4: Prepare resume for BERT
        input_ids, attention_masks, token_type_ids, tokens = prepare_resume_for_bert(
            cleaned_text, tokenizer
        )
        
        # Step 5: Extract entities using the model
        grouped_entities = extract_entities(
            model, input_ids, attention_masks, token_type_ids, tokens, idx2tag
        )
        
        # Step 6: Display the results
        display_results(grouped_entities)
        
        print("\nResume parsing complete!")
        
        return grouped_entities
    
    except Exception as e:
        print(f"Error in resume parsing process: {str(e)}")
        return {}

In [12]:
import os
if __name__ == "__main__":
    # Define paths
    current_dir = os.getcwd()  # Gets the current working directory
    resume_path = os.path.join(current_dir, "Aresume.pdf")
    model_path = os.path.join(current_dir, "eight_epoch.pt")
    # Check if the resume file exists
    if not os.path.exists(resume_path):
        print(f"Resume file not found: {resume_path}")
        print("Please make sure 'Aresume.pdf' is in the same directory as this script.")
    else:
        # Parse the resume
        parse_resume(resume_path, model_path)

Starting resume parsing process...
Converting PDF to text: C:\Users\user\Desktop\dyd\Aresume.pdf
PDF loaded successfully with 1 pages
Text extraction complete

Extracted Text Sample:
A KAIDEN KELLY
.NET Developer
+1-536-301-5054 Email linkedin.com/__victoria.johnson__
New York, NYC
SUMMARY
NET Developer with five years of experience developing web, batch, and business 
intelligenc...
Cleaning resume text...
Text cleaning complete

Cleaned Text Sample:
A KAIDEN KELLY .NET Developer 1-536-301-5054 Email linkedin.com__victoria.johnson__ New York, NYC SUMMARY NET Developer with five years of experience developing web, batch, and business intelligence s...
Loading AI model: C:\Users\user\Desktop\dyd\eight_epoch.pt
Model has 44 output classes
Error loading model: name 'BertConfig' is not defined
Failed to load model. Please check the model file.
